# Behavioral Trading Analysis — Beginner Walkthrough

This notebook rebuilds one question from FairValue Analytics with basic Pandas: **how do synthetic trade results differ after two consecutive losses?**

I am using synthetic portfolio data so the workflow can be public without exposing my personal accounts or results.

## 1. Load the trade table

A DataFrame is a table with rows and columns. Each row here represents one normalized completed trade.

In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/demo/trades.csv")
if not data_path.exists():
    data_path = Path("data/demo/trades.csv")

trades = pd.read_csv(data_path)
trades[["id", "account_id", "symbol", "entry_time", "gross_pnl"]].head()

## 2. Clean the columns

Dates and numbers sometimes arrive from CSV files as text. Converting them gives Pandas the correct type for sorting and calculations.

In [ ]:
trades["entry_time"] = pd.to_datetime(trades["entry_time"])
trades["exit_time"] = pd.to_datetime(trades["exit_time"])
trades["gross_pnl"] = pd.to_numeric(trades["gross_pnl"], errors="coerce").fillna(0)
trades = trades.sort_values(["account_id", "entry_time"]).reset_index(drop=True)

trades[["entry_time", "gross_pnl"]].dtypes

## 3. Calculate a few basic metrics

Win rate tells me how often a trade was positive. Average P&L also considers the size of wins and losses.

In [ ]:
basic_metrics = pd.Series({
    "trade_count": len(trades),
    "win_rate_percent": (trades["gross_pnl"] > 0).mean() * 100,
    "average_gross_pnl": trades["gross_pnl"].mean(),
    "total_gross_pnl": trades["gross_pnl"].sum(),
})
basic_metrics.round(2)

## 4. Create the `after_two_losses` feature

`groupby("account_id")` keeps account histories separate. `shift(1)` looks back one trade and `shift(2)` looks back two trades. The current trade is marked `True` only when both previous results in that account were negative.

In [ ]:
by_account = trades.groupby("account_id")["gross_pnl"]
trades["previous_pnl"] = by_account.shift(1)
trades["two_trades_ago_pnl"] = by_account.shift(2)
trades["after_two_losses"] = (
    (trades["previous_pnl"] < 0)
    & (trades["two_trades_ago_pnl"] < 0)
)

trades.loc[
    trades["after_two_losses"],
    ["id", "account_id", "entry_time", "gross_pnl", "previous_pnl", "two_trades_ago_pnl"],
]

## 5. Compare the two groups

This is descriptive analysis. It summarizes what happened in this synthetic sample; it does not prove that the loss streak caused the next outcome.

In [ ]:
comparison = (
    trades.groupby("after_two_losses")
    .agg(
        trade_count=("id", "count"),
        wins=("gross_pnl", lambda values: (values > 0).sum()),
        total_gross_pnl=("gross_pnl", "sum"),
        average_gross_pnl=("gross_pnl", "mean"),
    )
    .reset_index()
)
comparison["win_rate_percent"] = comparison["wins"] / comparison["trade_count"] * 100
comparison.round(2)

## 6. My interpretation

In the 21-trade synthetic example, the three trades following two consecutive losses win 33.3% of the time and total -$220 gross. The other 18 trades win 55.6% of the time and total +$2,650 gross. That makes a ten-minute lockout a reasonable rule to test.

The next data-science step is important: collect new trades after adopting the rule and evaluate those future observations. I should not use the same small sample both to invent the rule and to declare that it works.

## 7. What I would add next

- Results measured in risk units (R) instead of only dollars
- A before-versus-after comparison for the lockout rule
- Confidence intervals to show uncertainty
- More observations before attempting a predictive model